Building a Sales Brouchure Generator with OpenAI Chat Completion API

In [6]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [7]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print('There might be a probelm with the API key')

MODEL= 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [8]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

In [9]:
# Single-shot prompting example
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages = [
            {'role':'system', 'content':link_system_prompt},
            {'role':'user', 'content': get_links_user_prompt(url)}
        ],
        response_format = {'type':'json_object'}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [15]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [16]:
selected_links = select_relevant_links("https://huggingface.co")

In [17]:
selected_links

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Endpoint API', 'url': 'https://endpoints.huggingface.co'}]}

In [18]:
def fetch_page_and_all_relevant_info(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing page :\n\n{contents}\n## Relevant links:\n"
    for link in relevant_links['links']:
        result += f"\n\n ### Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [19]:
print(fetch_page_and_all_relevant_info("https://huggingface.co"))


## Landing page :

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5
Updated
4 days ago
•
167k
•
1.26k
MiniMaxAI/MiniMax-M2.5
Updated
about 22 hours ago
•
18.9k
•
670
Nanbeige/Nanbeige4.1-3B
Updated
4 days ago
•
21k
•
506
Qwen/Qwen3.5-397B-A17B
Updated
about 20 hours ago
•
495
openbmb/MiniCPM-SALA
Updated
6 days ago
•
3.67k
•
450
Browse 2M+ models
Spaces
Running
Reachy
290
Reachy Phone Home
📱
290
Phone focus companion for Reachy Mini
Running
748
Demo Playground
⚡
748
Free platform to access multiple AI models
Running
Featured
4.73k
Wan2.2 Animate
👁
4.73k
Wan2.2 Animate
Running
on
A100
Featured
401
ACE-Step v1.5
🎵
401
Music Generation Foundation Model v1.5
Running
on
Zero
MCP
783
Wan

In [37]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [38]:
def get_brouchure_user_prompt(company_name,url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_info(url)
    user_prompt = user_prompt[:5_000] # truncate if more than 5,000 characters
    return user_prompt


In [24]:
get_brouchure_user_prompt('HuggingFace',"https://huggingface.co")

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing page :\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\n4 days ago\n•\n167k\n•\n1.26k\nMiniMaxAI/MiniMax-M2.5\nUpdated\nabout 22 hours ago\n•\n18.9k\n•\n671\nNanbeige/Nanbeige4.1-3B\nUpdated\n4 days ago\n•\n21k\n•\n506\nQwen/Qwen3.5-397B-A17B\nUpdated\nabout 20 hours ago\n•\n495\nopenbmb/MiniCPM-SALA\nUpdated\n6 days ago\n•\n3.67k\n•\n450\nBrowse 2M+ models\nSpaces\nRunning\nReachy\n290\nReachy Phone Home\n📱\n290\nPhone 

In [39]:
def create_brouchure(company_name,url):
    response = openai.chat.completions.create(
        model= "gpt-4.1-mini",
        messages = [
            {'role':'system', 'content': brochure_system_prompt},
            {'role':'user', 'content': get_brouchure_user_prompt(company_name, url)}]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brouchure('HuggingFace', "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future — a leading collaboration platform where the machine learning (ML) community worldwide comes together to build, share, and innovate on models, datasets, and AI applications. It serves as the central hub for open-source ML, empowering engineers, scientists, and developers to create an open, ethical AI future.

---

## What We Offer

### A Collaborative Ecosystem

- **Models:** Access and share from a vast repository of **2 million+ ML models**, spanning modalities such as text, images, video, audio, and even 3D.
- **Datasets:** Explore and contribute to **500k+ datasets** critical for training and benchmarking models.
- **Spaces:** Launch and experience interactive AI applications with over **1 million+ hosted applications** available on the platform.

### The Hugging Face Hub

- A central place for ML practitioners to discover, experiment, and collaborate on open-source machine learning projects.
- Build and showcase your ML portfolio by sharing your work with a global community.
- Host unlimited public models, datasets, and apps to accelerate innovation.

### Enterprise & Compute Solutions

- Scalable paid compute resources designed to accelerate your ML workflows.
- Enterprise-grade security, access controls, and dedicated support tailored for teams and organizations.
- Solutions aimed at enhancing productivity and security for businesses working with AI.

---

## Company Culture & Community

- Hugging Face fosters a **vibrant, fast-growing community** of AI researchers, practitioners, and enthusiasts collaborating worldwide.
- The culture emphasizes **open collaboration**, transparency, and ethical AI development.
- The platform supports learning and mentorship for the next generation of machine learning professionals.

---

## Customers & Use Cases

- Hugging Face serves a broad spectrum of users — from independent ML researchers and hobbyists to large enterprises building AI-powered products.
- Users benefit from easily accessible resources to speed up AI development, from prototyping to production.
- Popular models and open datasets on the platform are used in natural language processing, computer vision, audio analysis, medical reasoning, and more.

---

## Careers & Opportunities

- Hugging Face seeks passionate individuals who want to be at the forefront of AI and machine learning innovation.
- Opportunities span engineering, research, community development, and enterprise support.
- Join a mission-driven company committed to openness, ethics, and advancing AI technology in a collaborative environment.

---

## Join Us

Explore AI apps, browse millions of models and datasets, or create your own on the Hugging Face platform.  

**Sign up today and accelerate your machine learning journey with the community that’s building the future of AI.**

[Visit Hugging Face](https://huggingface.co)

---

### Brand Colors

- Hugging Face Yellow: #FFD21E  
- Orange Accent: #FF9D00  
- Neutral Gray: #6B7280  

---

*Hugging Face – The Home of Machine Learning Collaboration*

In [40]:
def stream_brouchure(company_name,url):
    stream = openai.chat.completions.create(
        model ='gpt-4.1-nano',
        messages=[
            {'role':'system', 'content': brochure_system_prompt},
            {'role':'user', 'content': get_brouchure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [41]:
stream_brouchure('HuggingFace', "https://huggingface.co")

# Welcome to Hugging Face: The Future of AI, Made Fun and Friendly!

## Who Are We?
Imagine a place where machine learning wizards, data explorers, and AI adventurers come together to create the future. That’s us! Hugging Face is *the* community-powered platform built for those who believe AI should be open, collaborative, and downright exciting.

## What Do We Do?
- **Models Galore:** Jump into our maze of 2 million+ AI models – from text to video, and even 3D! Whether you’re just curious or ready to deploy, we’ve got you covered.
- **Datasets for Days:** Need the perfect data? Explore over 500,000 datasets—whether it’s medical data or math magic, we have it!
- **Spaces to Play:** Run and showcase your AI projects live in a friendly environment. Think of it as your personal AI sandbox, but better.

## Why Choose Us?
- **Community First:** We foster a vibrant, collaborative environment where everyone—from weekend hobbyists to AI pros—can learn, share, and grow.
- **Open Source Power:** Our open-source stack speeds up your projects, because who wants to wait?
- **All Modalities Welcome:** Text, images, videos, audio, and even 3D—if you can imagine it, you can build it here!

## Our Culture:
At Hugging Face, we believe AI should be *ethical, accessible, and above all* human-centered. We’re like the AI equivalent of a friendly neighborhood, continuously pushing the boundaries while keeping a cheeky sense of humor. Our motto? Build, share, and have fun doing it!

## Career & Team Spirit:
Think you’re a future AI superstar or a community enthusiast? Join us! We offer an environment where innovation meets camaraderie, with enterprise-grade security and benefits that won’t make you want to hide under the desk.

## Ready to Jump In?
Whether you’re here to collaborate, innovate, or just have some fun with AI, Hugging Face is your launching pad. Sign up today and help shape the future of AI—one model, one dataset, and one happy algorithm at a time!